# RNG Cost calculation notebook 

Start date: 09/08/2026
Author: Sahar H. El Abbadi

## Setup 

Import packages, load global variables and datasets

In [1]:
# Import packages 
import pandas as pd 
import pathlib

# Load variables 

# Physical constants
MMBTU_TO_MJ = 1055.06 # 1mmBtu = 1055.06MJ
HHV1 = 55.6 # MJ/KgCH4 (El Abbadi, Preprint)
P = 1 # Pressure(P) atm
T = 298 # Temperature(T) K
R = 0.0821 # Ideal gas constant(R) L.atm/mol.K

# Methane lower heating value - when water is not condensed 
KJ_PER_MOL_CH4 = 890.3   # kJ/mol, reported in Harrison et al., 2010
MW_CH4 = 16.04    # g/mol = 0.01604 kg/mol

# Convert kJ/mol to MJ/kg
METHANE_MJ_PER_KG = (KJ_PER_MOL_CH4 / 1000) / (MW_CH4 / 1000) # Result: 55.6 MJ / kg



In [21]:
# Load data 

# Facilities with AD in the United States 
ad_data_path = pathlib.Path("clean_data",
                       "ad_data.csv")

ad_data = pd.read_csv(ad_data_path)
print(ad_data)

         cwns_code                           facility state              city  \
0     3.000018e+10  BIGFORK COUNTY WATER & SEWER DIS.    MT           Bigfork   
1     4.700056e+10                    Gatlinburg WWTP    TN        Gatlinburg   
2     4.700203e+10         CLAIBORNE-NEW TAZEWELL STP    TN      New Tazewell   
3     2.200316e+10                       LOCKPORT STP    LA          Lockport   
4     2.200322e+10                  TRIUMPH-BURAS STP    LA             Buras   
...            ...                                ...   ...               ...   
3065  3.520090e+10             Santa Fe, City of WWTP    NM          Santa Fe   
3066  1.700049e+10       WOODRIDGE, GREEN VALLEY WWTP    IL         Woodridge   
3067  8.209000e+09                  Brighton, City of    CO          Brighton   
3068  6.002032e+09                     Oceanside WPCP    CA     San Francisco   
3069  6.008010e+09           IEUA Regional Plant No.4    CA  Rancho Cucamonga   

       latitude   longitude

## Helper functions 

In [3]:
## Standard units for material flows:
#  
# Flow rate: m3/day unless otherwise explicitly specified 
# Biogas production: kgCH4 / hr unless otherwise explicitly specified 
# RNG production: kgCH4 / hr unless otherwise explicitly specified 

#### Unit conversion equations 

## 1) Convert mmBTU of methane to kgCH4

def convert_mmBTU_to_kgCH4(mmBTU): 
    """Converts methane from units of mmBTU to kg CH4

    Parameters
    -----------
    mmBTU: float
        Natural gas / RNG energy content 

    Returns 
    -------
    float
        Calculated methane as kgCH4

    """

    # Convert from mmBTU to MJ 
    mj = mmBTU * MMBTU_TO_MJ

    # Convert from MJ to kg CH4
    kgCH4 = mj * (1/METHANE_MJ_PER_KG)

    return kgCH4

## 2) Convert kgCH4 to mmBTU

def convert_kgCH4_to_mmBTU(kgCH4): 
    """Converts methane from units of mmBTU to kg CH4

    Parameters
    -----------
    kgCH4: float
        Natural gas / biogas as kg CH4 

    Returns 
    -------
    float
        Calculated energy content of methane as mmBTU

    """
    # Convert from kg CH4 to MJ
    mj = kgCH4 * METHANE_MJ_PER_KG

    # Convert from MJ to mmBTU 
    mmBTU = mj * (1 / MMBTU_TO_MJ)

    return mmBTU



#### Mass flow equations

## 1) Calculate biogas production from facility flow rate 

def calc_biogas_from_ww_ad(flow_rate): # Calculates biogas flowrate in KgCH4/h
    """ Calculates biogas production rate for a wastewater treatment plant based on facility flow rate. 
    Source: El Abbadi et al, 2026; empirically derived 

    Parameters
    -----------
    flow_rate : float 
        Wastewater treatment plant flow rate in m3 / day

    Returns 
    -------
    float 
        Calculated biogas production rate in kgCH4 / hour
    """

    return flow_rate*0.00148




## 2) Calculate RNG production from biogas production 

def calc_rng_from_upgrading(biogas_input, efficiency=0.9): # Calculates RNG production in kgCH4/hr
    """ Calculate RNG produced from upgrading unit based on input efficiency 

    Parameters
    -----------
    biogas_input : float 
        Biogas production rate in kg CH4 / hour 
    
    efficiency : float 
        Upgrading efficiency, default value of 90%

    Returns 
    -------
    float 
        Calculated RNG production rate in kgCH4 / hour
    """

    return biogas_input * efficiency


## Calculate RNG production

For a given flow rate of a wastewater treatment plant (m3 / day), calculate expected RNG production 

In [ ]:
# Insert code here that uses the functions above to calculate RNG production from flow rate

# Input: flow rate
# Output: RNG production
# Why? We need RNG production rate as an input to Parker et al 

def calc_rng_production(flow_rate, efficiency=0.9): 
    """Calculates RNG production from biogas upgrading for a given input flow rate for a WWTP 
    
    Parameters
    ----------
    flow_rate : float 
        Wastewater treatment plant facility flow rate, units of m3 / day 
    
    efficiency : float
        Efficiency of biogas upgrading, value between 0 and 1

    Returns 
    -------
    float 
        Calculated RNG production rate, in kgCH4/hr

    """

    # Calculate biogas production based on flow rate
    biogas_produced = calc_biogas_from_ww_ad(flow_rate)

    # Calculate RNG production based on biogas production
    rng_produced = calc_rng_from_upgrading(biogas_produced, efficiency)

    return rng_produced



In [30]:
# Calculate RNG production for all facilities with AD, using data from El Abbadi et al., 2026

# Create a new column in ad_data with RNG production (kgCH4/hr) using functions defined above, with input of flow_m3_per_day from ad_data

# Relevant column: flow_m3_per_day
# Apply calc_rng_production to that column (flow, efficiency)

# make a new column in ad_data 
# The value of the column is going to be RNG product in kg CH4 / hr 
# We wrote a function calc_rng_production

# Make column with RNG production in kgCH4/hr
ad_data["rng_production_kgCh4_per_hr"] = ad_data["flow_m3_per_day"].apply(
    lambda x: calc_rng_from_upgrading(x, efficiency=0.9))

# Make column with RNG production in mmBTU/hr, using convert_kgCH4_to_mmBTU
ad_data["rng_production_mmBTU_per_hr"] = ad_data["rng_production_kgCh4_per_hr"].apply(
    lambda x: convert_kgCH4_to_mmBTU(x))

ad_data.head()

,cwns_code,facility,state,city,latitude,longitude,flow_mgd,median_ch4_kgco2e_day,median_n2o_kgco2e_day,median_co2_kgco2e_day,...,median_landapp_n2o_kgco2e_day,median_total_emission_kgco2e_day,treatment_train,has_chp,has_ad,flow_m3_per_day,electricity_cost,natural_gas_cost,rng_production_kgCh4_per_hr,rng_production_mmBTU_per_hr
0,3.000018e+10,BIGFORK COUNTY WATER & SEWER DIS.,MT,Bigfork,48.0638,-114.0831,0.50,639.708788,83.604170,71.876890,...,31.278405,832.366965,['*C1'],no,yes,1892.705892,0.0780,0.007195,1703.435303,89.614956
1,4.700056e+10,Gatlinburg WWTP,TN,Gatlinburg,35.7355,-83.5227,3.00,3841.681050,499.533687,428.483149,...,0.000000,6750.936402,['*A1'],no,yes,11356.235352,0.0623,0.005031,10220.611817,537.689735
2,4.700203e+10,CLAIBORNE-NEW TAZEWELL STP,TN,New Tazewell,36.4637,-83.5608,0.65,465.995591,108.884187,92.571204,...,0.000000,1083.537517,"['*C1', '*C3']",no,yes,2460.517660,0.0623,0.005031,2214.465894,116.499443
3,2.200316e+10,LOCKPORT STP,LA,Lockport,29.6419,-90.5464,0.35,447.996137,58.400925,50.151762,...,0.000000,777.952415,"['*A1', '*C1']",no,yes,1324.894124,0.0588,0.003516,1192.404712,62.730469
4,2.200322e+10,TRIUMPH-BURAS STP,LA,Buras,29.3416,-89.5167,2.50,3198.543941,418.020850,359.384450,...,0.000000,5433.666009,['*C1'],no,yes,9463.529460,0.0588,0.003516,8517.176514,448.074779


## Calculate RNG upgrading cost 

In [ ]:
# Cost estimate using Parker et al., 2017

# Parker et al Table 1 reports: 
## Upgrading to pipeline 
## Capital cost = 1_370_000 * RNG_Flow^0.56
## Annual O&M = 101_625 * RNG_Flow^0.81

# RNG refers to RNG produced and is in units of mmBtu / hr 

# Function for Parker et al 

def calc_capex_parker(biogas_produced_kgCH4): 
    """ Calculate capital cost of biogas upgrading to RNG for a given biogas production rate 

    Parameters
    -----------
    biogas_produced_kgCH4 : float 
        RNG production rate in kg CH4 / hour 

    Returns 
    -------
    float 
        Calculated capital cost in USD 
    """

    # 1) Convert RNG in kgCH4/hr to mmBTU per hr 
    biogas_produced_mmBTU = convert_kgCH4_to_mmBTU(biogas_produced_kgCH4)

    # 2) Convert biogas produced (MMBtu) to RNG produced. Use default efficiency value for now 
    rng_produced_mmBTU = calc_rng_from_upgrading(biogas_produced_mmBTU)

    # 3) Apply Parker et al equation 

    upgrading_capex = 1_370_000 * rng_produced_mmBTU^0.56

    return upgrading_capex